In [0]:
import os, sys, time
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, gold, group, privacy

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
print(cfg["code"], env, "group, reads", group.source(cfg, env) + ".gold")

In [0]:
start = time.time()
built = group.build(spark, cfg, env)
for name, rows in built.items():
    print(f"{name:22} {rows:>6} rows")

masks = privacy.apply(spark, cfg, env)            # group tables get their masks before anyone reads them
print("masks", masks)

merge = group.merge_control(spark, cfg, env, built)   # BHG never waits: missing FQF days go to the backlog
print("merge", {k: str(v) for k, v in merge.items() if k != "tables"})

print("TOTAL", built, f"in {time.time() - start:.0f}s")
common.log_run(spark, cfg, "group", "success", sum(built.values()), str(built))